In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env


def reward_function(history):   
    current_val = history["portfolio_valuation", -1]
    last_val = history["portfolio_valuation", -2]
    reward = (current_val / last_val) - 1
    
    return reward


def preprocess(df):
    df = df.copy()
    df = df.sort_index()
    
    # Price Position
    # Bollinger band (price according to min and max)
    sma = df["close"].rolling(window=20).mean()
    std = df["close"].rolling(window=20).std()
    upper = sma + (2 * std)
    lower = sma - (2 * std)
    df["feature_bb_pos"] = (df["close"] - lower) / (upper - lower) # Position (0 = low, 1 = high, >1 = higher than before)
    df["feature_bb_width"] = (upper - lower) / df["close"]
    
    # Trend
    # Distance to Moving Average 20
    df["ma_20"] = df["close"].rolling(window=20).mean()
    df["feature_dist_ma_20"] = df["close"] / df["ma_20"] - 1
    # Distance to Moving Average 50
    df["ma_50"] = df["close"].rolling(window=50).mean()
    df["feature_dist_ma_50"] = df["close"] / df["ma_50"] - 1
    # Distance to Moving Average 200
    df["ma_200"] = df["close"].rolling(window=200).mean()
    df["feature_dist_ma_200"] = df["close"] / df["ma_200"] - 1
    
    
    # Trend speed
    df["feature_log_ret"] = np.log(df["close"] / df["close"].shift(1))
    
    # Trend Strength
    # ADX Simplified
    high_low = df["high"] - df["low"]
    high_close = np.abs(df["high"] - df["close"].shift())
    low_close = np.abs(df["low"] - df["close"].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    atr = true_range.rolling(14).mean() + 1e-10
    # Directional Movement
    up_move = df["high"] - df["high"].shift()
    down_move = df["low"].shift() - df["low"]
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
    plus_di = 100 * (pd.Series(plus_dm, index=df.index).rolling(14).mean() / atr)
    minus_di = 100 * (pd.Series(minus_dm, index=df.index).rolling(14).mean() / atr)
    # DX et ADX
    sum_di = plus_di + minus_di + 1e-10
    dx = 100 * np.abs(plus_di - minus_di) / sum_di
    df["feature_adx"] = dx.rolling(14).mean() / 100.0

    # Volatility
    df["feature_std_deviation"] = df["close"].rolling(window=20).std() / df["close"]
    df["feature_candle_range"] = (df["high"] - df["low"]) / df["close"]

    # Momentum
    # RSI
    delta = df["close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df["rsi"] = 100 - (100 / (1 + rs))
    df["feature_rsi_norm"] = df["rsi"] / 100.0
    
    # Volume
    # Test if Volume == 0
    if "volume" not in df.columns or (df["volume"] == 0).all():
        df["volume"] = 1.0 # Value with no impact to prevent crash
        has_volume = False
    else:
        df["volume"] = df["volume"].replace(0, 1e-5) # Replace 0 with low values
        has_volume = True
    # Calculations
    if has_volume:
        # Volume Relative Strength
        df["vol_ma_20"] = df["volume"].rolling(window=20).mean()
        df["feature_vol_rel"] = (df["volume"] / df["vol_ma_20"] - 1).fillna(0)
        # OBV Slope
        df["obv"] = (np.sign(df["close"].diff()) * df["volume"]).fillna(0).cumsum()
        df["feature_obv_slope"] = df["obv"].pct_change(5).fillna(0)
    else:
        df["feature_vol_rel"] = 0.0
        df["feature_obv_slope"] = 0.0
        
    # Clean up
    df.dropna(inplace=True) # Remove NaN
    df.replace([np.inf, -np.inf], 0, inplace=True) # Remove infinite values due to calculations
    
    return df


base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess,
    portfolio_initial_value=1_000,
    trading_fees=0.1/100,
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_function,
)

In [2]:
from gym_trading_env.wrapper import DiscreteActionsWrapper

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)
print("obs :", obs)
print("reward :", reward)
print("terminated :", terminated)
print("truncated :", truncated)
print("info :", info)

obs : [ 1.0702626   0.0313102   0.01817964 -0.00285825 -0.02663784  0.00999056
  0.13406911  0.00782755  0.01124747  0.63444    -0.16164564 -0.11843392
  2.          1.9803302 ]
reward : 0.019009650469298478
terminated : False
truncated : False
info : {'idx': 1, 'step': 1, 'date': np.datetime64('2020-08-26T15:00:00.000000000'), 'position_index': 3, 'position': 2, 'real_position': np.float64(1.9803302396267133), 'data_low': 386.42, 'data_obv': -35378.98924663001, 'data_date_close': Timestamp('2020-08-26 16:00:00'), 'data_rsi': 63.443998540678606, 'data_open': 386.42, 'data_ma_200': 400.99154999999996, 'data_volume': 1674.62301852, 'data_ma_20': 383.341, 'data_close': 390.31, 'data_ma_50': 391.42879999999997, 'data_high': 390.81, 'data_vol_ma_20': 1997.5121428824998, 'portfolio_valuation': np.float64(1019.0096504692984), 'portfolio_distribution_asset': np.float64(5.170186839424558), 'portfolio_distribution_fiat': 0, 'portfolio_distribution_borrowed_asset': 0, 'portfolio_distribution_borr

In [155]:
from stable_baselines3 import DQN

model = DQN(
    "MlpPolicy", 
    env, 
    verbose=1, 
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [156]:
# Model Training on ./data/*.pkl files
model.learn(total_timesteps=300_000, log_interval=10)

-0.0006648454333474385
-0.0009999999999998899
0.0
0.0
0.0
-0.00352595537814826
-0.0012197983092359088
-0.0018978848031723228
-0.0010008001850511095
0.0
-0.00023930495241730743
-0.0020933638528359833
-0.0019927625288768125
-0.0008914467981756546
-0.0004894479923763795
-0.003070920568311375
-0.0009999916662986053
-6.15065837502593e-05
-0.0007828036859414134
-0.0012504079922021072
-0.0009091154405108659
-0.0001411949215877062
0.0004334607375275912
-0.0019903949094717266
-0.001223910214220858
-0.001001451821463939
0.0
-0.0012165117642556345
-0.0023308898019341884
-0.0010016690000831385
-0.0012241615977938158
-0.005374342281733169
-0.0019940365335960886
0.0
-0.0010000000000000009
-0.0010000000000000009
-0.0022972468702833204
-0.0010000000000000009
0.0004067321161118631
-0.000279711774696878
-0.0023119420176355154
-0.0009999916761377348
-0.0005687273927048597
-0.0014680770541314336
-0.000999938530413802
-6.660476839159823e-05
-0.0033181282406555823
0.0001081172582388934
-0.002315290058039765

KeyboardInterrupt: 

In [ ]:
# Test
obs, info = env.reset()
done = False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    
    obs, reward, terminated, truncated, info = env.step(int(action))
    done = terminated or truncated

# Logs for rendering
env.unwrapped.save_for_render(dir="render_logs")

Market Return : 104.07%   |   Portfolio Return : -87.38%   |   
